# Part 3 — No-show classification and threshold selection

Target is `no_show`; exclude patient/appointment/clinic identifiers and leakage fields. Split using test_size=0.2 and random_state=42. Reminder and prior_no_shows are known at the prediction time, as defined by the brief.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score,confusion_matrix
from sklearn.impute import SimpleImputer
df=pd.read_csv('appointments_clean.csv')
df['reminder_sent']=df.reminder_sent.map({'Yes':1,'No':0})
features=['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent','booking_channel','visit_type']
X=df[features]; y=df.no_show
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
num=['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent']; cat=['booking_channel','visit_type']
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
print('Rows',len(df),'features',features,'train/test',len(X_train),len(X_test),'base rate',y.mean())

**Task 3.2 — Logistic regression.** Fit only on training rows; report test accuracy, confusion matrix, precision, recall and ROC-AUC. Coefficients are association, not causal effects.

In [ ]:
lr=Pipeline([('prep',pre),('model',LogisticRegression(max_iter=1000))])
lr.fit(X_train,y_train); p_lr=lr.predict(X_test); prob_lr=lr.predict_proba(X_test)[:,1]
print('Logistic accuracy',accuracy_score(y_test,p_lr),'precision',precision_score(y_test,p_lr,zero_division=0),'recall',recall_score(y_test,p_lr,zero_division=0),'ROC-AUC',roc_auc_score(y_test,prob_lr))
print('Confusion matrix:\n',confusion_matrix(y_test,p_lr))

**Task 3.3 — Decision tree.** Compare unrestricted and depth-3 tree for train/test accuracy to show overfitting.

In [ ]:
tree=Pipeline([('prep',pre),('model',DecisionTreeClassifier(random_state=42))])
tree.fit(X_train,y_train)
tree3=Pipeline([('prep',pre),('model',DecisionTreeClassifier(max_depth=3,random_state=42))])
tree3.fit(X_train,y_train)
print('Tree full train/test:',tree.score(X_train,y_train),tree.score(X_test,y_test))
print('Tree depth3 train/test:',tree3.score(X_train,y_train),tree3.score(X_test,y_test))

**Task 3.4 — Random forest and comparison.** Compare accuracy, recall, and ROC-AUC. Feature importance for forest is printed below.

In [ ]:
rf=Pipeline([('prep',pre),('model',RandomForestClassifier(n_estimators=100,random_state=42))])
rf.fit(X_train,y_train); p_rf=rf.predict(X_test); prob_rf=rf.predict_proba(X_test)[:,1]
models={'Logistic':(p_lr,prob_lr),'Tree depth 3':(tree3.predict(X_test),tree3.predict_proba(X_test)[:,1]),'Random Forest':(p_rf,prob_rf)}
for name,(pred,prob) in models.items(): print(name,'accuracy',accuracy_score(y_test,pred),'precision',precision_score(y_test,pred,zero_division=0),'recall',recall_score(y_test,pred,zero_division=0),'ROC-AUC',roc_auc_score(y_test,prob))
print('Forest importances:')
for f,i in sorted(zip(features,rf.named_steps['model'].feature_importances_),key=lambda x:x[1],reverse=True): print(f,round(i,4))

**Task 3.5 — CV comparison, thresholding, and test-once.** Use training-only cross-validation to choose a model; threshold selection can be based on the trade-off between recall and precision, but do not use the test set to tune threshold. The brief specifies a final logistic-regression CV run and comparison to tree/forest; sample below.

In [ ]:
cv_scores=cross_val_score(lr,X_train,y_train,cv=5,scoring='roc_auc')
print('Logistic train 5-fold ROC-AUC mean',cv_scores.mean(),'std',cv_scores.std())
# Example operational threshold chosen without tuning on test; illustrate 0.30 threshold
threshold=0.30
chosen_pred=(prob_lr>=threshold).astype(int)
print('At threshold',threshold,'precision',precision_score(y_test,chosen_pred,zero_division=0),'recall',recall_score(y_test,chosen_pred,zero_division=0),'F1',__import__('sklearn.metrics').metrics.f1_score(y_test,chosen_pred))
# final exported score file (test subset only)
out=df.loc[X_test.index,['appointment_id']].copy(); out['risk_score']=prob_lr; out['actual_no_show']=y_test.values; out['risk_flag']=chosen_pred
out.to_csv('test_risk_scores.csv',index=False); out.head()